In [ ]:
import os
import json as _json
from pathlib import Path
from dataclasses import dataclass, field
from typing import cast
from typing import List, Optional, Dict, Any, Tuple, Union, Callable, Set
import pandas as pd
import polars as pl
import io
import numpy as np
from types import SimpleNamespace
from polars.testing import assert_frame_equal as pl_assert_frame_equal
print('pandas:', pd.__version__, ' polars:', pl.__version__)

In [ ]:
# ── Fixtures ────────────────────────────────────────────────────────────────

# --- pairwise_loc_colname ---
FIX_PAIRWISE_LOC_COLNAME_COL_VALUE_LIST = ["paper_A", "paper_B"]
FIX_PAIRWISE_LOC_COLNAME_COLNAME = "title"
FIX_PAIRWISE_LOC_COLNAME_DOCUMENT_ID_1 = 101
FIX_PAIRWISE_LOC_COLNAME_DOCUMENT_ID_2 = 102
FIX_PAIRWISE_LOC_COLNAME_ROW_VALUE_LIST = ["paper_A", "paper_B"]

# --- pairwise_loc_item ---
FIX_PAIRWISE_LOC_ITEM_COL_EMBEDDING = np.array([0.1, 0.2, 0.3])
FIX_PAIRWISE_LOC_ITEM_DOCUMENT_ID_1 = 101
FIX_PAIRWISE_LOC_ITEM_DOCUMENT_ID_2 = 102
FIX_PAIRWISE_LOC_ITEM_ROW_EMBEDDING = np.array([0.1, 0.2, 0.3])

print("✅ Fixtures loaded")
df = pd.DataFrame({"title":["paper_A","paper_B","paper_C"]}, index=[101,102,103])


In [ ]:
# ── Before wrappers (verbatim pandas) ───────────────────────────────────────

def before_pairwise_loc_colname(col_value_list, colname, document_id_1, document_id_2, row_value_list):
    row_value_list: list[str] = df.loc[document_id_1, colname]
    col_value_list: list[str] = df.loc[document_id_2, colname]
    return row_value_list, col_value_list

def before_pairwise_loc_item(col_embedding, document_id_1, document_id_2, row_embedding):
    row_embedding: EmbeddingVector = df.loc[document_id_1].item()
    col_embedding: EmbeddingVector = df.loc[document_id_2].item()
    return None


In [ ]:
# ── Generated wrappers (experiment-generated Polars) ─────────────────────────

def gen_pairwise_loc_colname(col_value_list, colname, document_id_1, document_id_2, row_value_list):

    row_value_list: list[str] = (
        df.filter(pl.col("document_id") == document_id_1)
        .get_column(colname)
        .to_list()
    )
    col_value_list: list[str] = (
        df.filter(pl.col("document_id") == document_id_2)
        .get_column(colname)
        .to_list()
    )
    return row_value_list, col_value_list

def gen_pairwise_loc_item(col_embedding, document_id_1, document_id_2, row_embedding):
    row_embedding: EmbeddingVector = df.filter(pl.col("document_id") == document_id_1).select(pl.all().exclude("document_id")).item()
    col_embedding: EmbeddingVector = df.filter(pl.col("document_id") == document_id_2).select(pl.all().exclude("document_id")).item()
    return None


In [ ]:
# ── Comparison helper ───────────────────────────────────────────────────────
def _index_is_trivial(idx):
    # Unnamed + integer-valued covers both a fresh RangeIndex and the leftover
    # positional index after filtering/boolean-masking a RangeIndex-based frame
    # (pandas downgrades RangeIndex to a plain Int64Index on filter, but it's
    # still just leftover row positions, not real data). A set_index(...)
    # always carries the original column's name, so any genuinely meaningful
    # index is caught by the "name is not None" branch.
    return idx.name is None and pd.api.types.is_integer_dtype(idx.dtype)


def _to_pl(r):
    if isinstance(r, pl.DataFrame): return r
    if isinstance(r, pd.DataFrame): return pl.from_pandas(r.reset_index(drop=True) if _index_is_trivial(r.index) else r.reset_index())
    if isinstance(r, pd.Series): return pl.from_pandas(r.to_frame().reset_index(drop=True) if _index_is_trivial(r.index) else r.to_frame().reset_index())
    return None

def compare(before_result, gen_result, label, check_row_order=False):
    raw_label = str(label)
    label_parts = raw_label.strip().split()
    is_l3 = bool(label_parts and label_parts[0].upper() == "L3")
    layer = "L3" if is_l3 else "L2"
    kind = "edge" if is_l3 else "equivalence"
    if is_l3:
        label_parts = label_parts[1:]
        if label_parts and label_parts[0].lower() in ("edge", "branch"):
            label_parts = label_parts[1:]
        display_label = " ".join(label_parts)
    else:
        display_label = raw_label

    left  = _to_pl(before_result.collect() if isinstance(before_result, pl.LazyFrame) else before_result)
    right = _to_pl(gen_result.collect() if isinstance(gen_result, pl.LazyFrame) else gen_result)
    if left is None and right is None:
        print(f"⚠️  {layer} {kind} {display_label}: both sides non-DataFrame (no output to compare)")
        return
    if left is None or right is None:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — one side returned DataFrame, other did not")
        return
    left_cols, right_cols = set(left.columns), set(right.columns)
    if left_cols != right_cols:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — column sets differ (before-only={left_cols - right_cols}, gen-only={right_cols - left_cols})")
        return
    common = list(left.columns)
    try:
        pl_assert_frame_equal(left.select(common), right.select(common),
                              check_dtypes=False, check_row_order=check_row_order)
        print(f"✅ {layer} {kind} {display_label}: MATCH")
    except Exception as e:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — {e}")


In [ ]:
# === Tests: pairwise_loc_colname ===
def _run_pairwise(frame, fn):
    global df; _old=df
    try: df=frame; return fn(FIX_PAIRWISE_LOC_COLNAME_COL_VALUE_LIST, FIX_PAIRWISE_LOC_COLNAME_COLNAME, FIX_PAIRWISE_LOC_COLNAME_DOCUMENT_ID_1, FIX_PAIRWISE_LOC_COLNAME_DOCUMENT_ID_2, FIX_PAIRWISE_LOC_COLNAME_ROW_VALUE_LIST)
    finally: df=_old
try:
    _r=_run_pairwise(pl.DataFrame({"document_id":[101,102,103],"title":["paper_A","paper_B","paper_C"]}),gen_pairwise_loc_colname); print("✅ L1 smoke gen_pairwise_loc_colname: OK, type=",type(_r).__name__)
except Exception as _e: print(f"❌ L1 smoke gen_pairwise_loc_colname: {type(_e).__name__}: {_e}")
try:
    _rb=_run_pairwise(pd.DataFrame({"title":["paper_A","paper_B","paper_C"]},index=[101,102,103]),before_pairwise_loc_colname); print("✅ L1 smoke before_pairwise_loc_colname: OK")
except Exception as _e: print(f"❌ L1 smoke before_pairwise_loc_colname: {type(_e).__name__}: {_e}")
try:
    _rb=_run_pairwise(pd.DataFrame({"title":["paper_A","paper_B","paper_C"]},index=[101,102,103]),before_pairwise_loc_colname); _rg=_run_pairwise(pl.DataFrame({"document_id":[101,102,103],"title":["paper_A","paper_B","paper_C"]}),gen_pairwise_loc_colname)
    if _rb==_rg: print("✅ L2 equivalence pairwise_loc_colname captured locals: MATCH")
    else: print(f"❌ L2 equivalence pairwise_loc_colname captured locals: MISMATCH — before={_rb!r}, gen={_rg!r}")
except Exception as _e: print(f"❌ L2 equivalence pairwise_loc_colname: setup error — {type(_e).__name__}: {_e}")

# L3 edge - missing identifiers must be rejected by both implementations.
try:
    _before_exc = _gen_exc = None
    try:
        _run_pairwise(pd.DataFrame({"title": pd.Series(dtype="object")}, index=pd.Index([], dtype="int64")), before_pairwise_loc_colname)
    except Exception as _e:
        _before_exc = _e
    try:
        _run_pairwise(pl.DataFrame(schema={"document_id": pl.Int64, "title": pl.String}), gen_pairwise_loc_colname)
    except Exception as _e:
        _gen_exc = _e
    if _before_exc is not None and _gen_exc is not None and not isinstance(_gen_exc, (SyntaxError, NameError)):
        print(f"✅ L3 edge pairwise_loc_colname missing ids: both sides reject (before={type(_before_exc).__name__}, gen={type(_gen_exc).__name__})")
    else:
        print(f"❌ L3 edge pairwise_loc_colname missing ids: MISMATCH — before={_before_exc}, gen={_gen_exc}")
except Exception as _e:
    print(f"❌ L3 edge pairwise_loc_colname: {type(_e).__name__}: {_e}")
